In [0]:
# ============================================================
# 0. CONFIG PATH
# ============================================================
dbutils.widgets.text("path","")
config_path = dbutils.widgets.get("path")
# ============================================================
# 1. IMPORTS
# ============================================================
import json
import pyspark.sql.functions as f
from common_utils.generic_functions import  casting
from common_utils.logging import get_logger

# ============================================================
# 2. LOGGER
# ============================================================
logger = get_logger("dim_customers.ipynb")
logger.info("Session for dim_customer_gold started")

# ============================================================
# 3. READ CONFIG
# ============================================================

logger.info(
    "Reading Gold configuration from %s",config_path
)
with open(config_path,"r") as file:
    config = json.load(file)

logger.info("Gold configuration loaded successfully")

# ============================================================
# 4. SOURCE / TARGET CONFIGURATION
# ============================================================
source_config = config["source"]
target_config= config["target"]

source_name = config["source"]["source_name"]
source_table = config["source"]["source_table"]
target_table = config["target"]["target_table"]

logger.info(
    "[%s] Source table: %s",
    source_name,
    source_table
)

logger.info(
    "[%s] Target table: %s",
    source_name,
    target_table
)


# ============================================================
# 5. READ SOURCE TABLE
# ============================================================

logger.info(
    "[%s] Reading customers from %s",
    source_name,
    source_table
)

customers_df = spark.read.table(source_table)

logger.info(
    "[%s] Customers table loaded successfully",
    source_name
)

# ============================================================
# 6. TRANSFORM CUSTOMER
# ============================================================


def transform_customer(df,config):
    logger.info(
        "[%s] Starting dim_customer transformation",
        source_name
    )
    

    # --------------------------------------------------------
    # Selecting required columns
    # --------------------------------------------------------

    df = df.select("customer_id","first_name","last_name","loyalty_segment","valid_from","last_update_timestamp")

    # --------------------------------------------------------
    # Cleaning names
    # --------------------------------------------------------
    df = (df.withColumn("first_name",f.trim(f.col("first_name"))).withColumn("last_name",f.trim(f.col("last_name"))))


    # --------------------------------------------------------
    # Create full name
    # --------------------------------------------------------
    df = df.withColumn("full_name",f.trim(f.concat_ws(" ",f.col("first_name"),f.col("last_name"))))

    # --------------------------------------------------------
    # Derive customer type
    #
    # Keep business meaning configurable rather than assuming
    # what loyalty segment 0/1/2 means.
    # --------------------------------------------------------
    
    df = df.withColumn("customer_type",f.when(f.col("loyalty_segment").isNull(),f.lit(config["segment_values"]["default_customer_type"])).otherwise(f.concat(f.lit("Segment_"),f.col("loyalty_segment").cast("string"))))

    # --------------------------------------------------------
    # Created date
    # --------------------------------------------------------

    df = df.withColumn("created_date", f.to_date(f.col("valid_from").cast("timestamp")))

    # --------------------------------------------------------
    # Last updated
    # --------------------------------------------------------

    df = df.withColumn("last_updated",f.to_timestamp(f.col("last_update_timestamp")))
    # --------------------------------------------------------
    # Cast final columns
    # --------------------------------------------------------

    df = casting(df, config["cast_columns"])

    df = df.select(
        *config["select_columns"]
    )
    df = (
        df
        .orderBy(
            f.col("last_updated").desc()
        )
        .dropDuplicates(
            ["customer_id"]
        )
    )
    logger.info(
        "[%s] dim_customer transformation completed",
        source_name
    )

    return df

# ============================================================
# 7. CREATE DIM CUSTOMER
# ============================================================

logger.info(
    "[%s] Starting dim_customer creation",
    source_name
)


#creating dim customer
dim_customer = transform_customer(customers_df,config)


# ============================================================
# 8. VALIDATE DATA
# ============================================================
logger.info(
    "[%s] Validating dim_customer",
    source_name
)

records_count=dim_customer.count()
if records_count==0:
    raise ValueError("no record found in Dim_customer")

duplicate_count = (
    dim_customer
    .groupBy("customer_id")
    .count()
    .filter(f.col("count") > 1)
    .count()
)
if duplicate_count > 0:
    raise ValueError ("Duplicae detected")


# ============================================================
# 9. DISPLAY
# ============================================================
dim_customer.display()




# ============================================================
# 10. CREATE GOLD SCHEMA
# ============================================================
logger.info(
    "[%s] Creating Gold schema if required",
    source_name
)

spark.sql("""
    CREATE SCHEMA IF NOT EXISTS 
    retail_data_platform.gold
    
          """)
# ============================================================
# 11. WRITE GOLD TABLE
# ============================================================

logger.info(
    "[%s] Writing dim_customer to %s",
    source_name,
    target_table
)

(dim_customer.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table ))

logger.info(
    "[%s] Gold table %s written successfully",
    source_name,
    target_table
)

# ============================================================
# 12. FINAL VALIDATION
# ============================================================

final_count = (
    spark.read
    .table(target_table)
    .count()
)

logger.info(
    "[%s] Final dim_customer record count: %s",
    source_name,
    final_count
)

logger.info(
    "[%s] dim_customer pipeline completed successfully",
    source_name
)

